# Candle Notebook: Minimal Candle + Image Save Test

This minimal notebook validates basic tensor creation and image saving without any GUI.
We standardize the environment so it runs from any folder:
- Set the working directory to the repo root
- Use an image store directory at `images_store`
- Rely on `candle-notebooks` crate (crates.io) for helpers, not path dependencies

The next cell declares the dependencies and performs a unified setup.

In [ ]:
:toolchain stable
:dep candle-notebooks = "0.1.0"
:dep image = "0.24"

// Unified setup: dependency declarations, imports, and initialization
use candle_notebooks as nb;
use nb::{Device, Tensor};
use candle_notebooks::ah as anyhow;
use anyhow::Result;

nb::set_notebook_cwd().expect("failed to set notebook cwd");
nb::set_image_store_rel_dir("images_store").unwrap();
std::fs::create_dir_all("images_store").ok();

let device = Device::Cpu;
println!("Toolchain: stable | Device: {:?}", device);
println!("CWD: {}", std::env::current_dir().unwrap().display());
println!("Image store: images_store/");

Notebook CWD set to repository root: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/research/notebooks/candle_notebooks
Toolchain: stable | Device: Cpu
CWD: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/research/notebooks/candle_notebooks
Image store: images_store/


Toolchain: stable

In [5]:
// Minimal candle test without GUI: build an RGB gradient and save with image crate.
use anyhow::Result;

fn save_image<P: AsRef<std::path::Path>>(img: &Tensor, p: P) -> Result<()> {
    let p = p.as_ref();
    let (c,h,w) = img.dims3()?; if c!=3 { anyhow::bail!("expected (3,H,W)"); }
    let img2 = img.permute((1,2,0))?.flatten_all()?;
    let pixels = img2.to_vec1::<u8>()?;
    let buffer: image::ImageBuffer<image::Rgb<u8>, Vec<u8>> = image::ImageBuffer::from_raw(w as u32,h as u32,pixels).ok_or_else(|| anyhow::anyhow!("raw to image"))?;
    buffer.save(p)?; Ok(())
}

// Create a simple RGB gradient in CHW layout as u8
let h: usize = 140; let w: usize = 200; let device: Device = Device::Cpu;
let mut r: Vec<u8> = Vec::with_capacity(h*w);
let mut g: Vec<u8> = Vec::with_capacity(h*w);
let mut b: Vec<u8> = Vec::with_capacity(h*w);
for y in 0..h { for x in 0..w {
    let fx = x as f32 / (w as f32 - 1.0);
    let fy = y as f32 / (h as f32 - 1.0);
    r.push((fx * 255.0).round() as u8);
    g.push((fy * 255.0).round() as u8);
    b.push((((fx + fy) / 2.0) * 255.0).round() as u8);
}}
let rt: Tensor = Tensor::from_vec(r, (h,w), &device)?;
let gt: Tensor = Tensor::from_vec(g, (h,w), &device)?;
let bt: Tensor = Tensor::from_vec(b, (h,w), &device)?;
let rgb: Tensor = Tensor::stack(&[&rt,&gt,&bt],0)?;
std::fs::create_dir_all("images_store")?;
save_image(&rgb, "images_store/test_gradient2.png")?;
println!("Saved images_store/test_gradient2.png");

Saved images_store/test_gradient2.png
